In [177]:
pip install pandas numpy

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'd:\Student_Placement_Readiness_Predictor\venv\Scripts\python.exe -m pip install --upgrade pip' command.


In [178]:
import pandas as pd
import numpy as np

In [179]:
pip install matplotlib seaborn

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'd:\Student_Placement_Readiness_Predictor\venv\Scripts\python.exe -m pip install --upgrade pip' command.


In [180]:
import matplotlib.pyplot as plt
import seaborn as sns

In [181]:
%matplotlib inline


In [182]:
df = pd.read_csv('../data/raw/students_dataset.csv')

In [183]:
df.head()

,branch,gender,college_type,study_mode,cgpa,dsa_problems_solved,mock_test_score,projects_count,resume_score,study_hours_per_day,attendance_percentage,sleep_hours,stress_level,internship,core_subjects_confidence,core_subjects_covered_count,placement_status
0,EEE,Female,Tier1,Coaching,5.63,380,70,3,59,1.9,56,7.6,3,0,3,3,Needs Improvement
1,MECH,Female,Tier1,Mixed,5.72,290,61,1,96,8.9,65,4.0,2,0,1,4,Needs Improvement
2,ECE,Female,Tier2,Self,6.11,579,44,3,89,5.7,68,7.0,1,0,4,0,Placement Ready
3,MECH,Female,Tier3,Coaching,6.42,532,91,4,80,8.4,88,8.2,9,0,3,3,Placement Ready
4,MECH,Female,Tier1,Mixed,8.36,236,56,5,58,8.7,55,5.8,4,1,2,3,Placement Ready


In [184]:
df.tail()

,branch,gender,college_type,study_mode,cgpa,dsa_problems_solved,mock_test_score,projects_count,resume_score,study_hours_per_day,attendance_percentage,sleep_hours,stress_level,internship,core_subjects_confidence,core_subjects_covered_count,placement_status
995,IT,Female,Tier2,Mixed,6.64,258,86,1,99,1.3,80,5.5,6,1,4,4,Placement Ready
996,CSE,Female,Tier1,Self,6.70,146,97,6,67,2.3,86,6.7,9,1,2,4,Placement Ready
997,CSE,Male,Tier2,Coaching,6.69,123,81,6,72,6.5,74,4.1,5,0,1,0,Needs Improvement
998,EEE,Female,Tier1,Self,5.65,240,66,4,59,2.6,65,6.6,8,1,5,1,Needs Improvement
999,ECE,Female,Tier1,Self,9.49,105,37,5,60,3.0,87,7.2,1,0,1,3,Placement Ready


In [185]:
df.shape

(1000, 17)

In [186]:
df.isna().sum()

branch                         0
gender                         0
college_type                   0
study_mode                     0
cgpa                           0
dsa_problems_solved            0
mock_test_score                0
projects_count                 0
resume_score                   0
study_hours_per_day            0
attendance_percentage          0
sleep_hours                    0
stress_level                   0
internship                     0
core_subjects_confidence       0
core_subjects_covered_count    0
placement_status               0
dtype: int64

In [187]:
df.duplicated().sum()

np.int64(0)

In [188]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 17 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   branch                       1000 non-null   object 
 1   gender                       1000 non-null   object 
 2   college_type                 1000 non-null   object 
 3   study_mode                   1000 non-null   object 
 4   cgpa                         1000 non-null   float64
 5   dsa_problems_solved          1000 non-null   int64  
 6   mock_test_score              1000 non-null   int64  
 7   projects_count               1000 non-null   int64  
 8   resume_score                 1000 non-null   int64  
 9   study_hours_per_day          1000 non-null   float64
 10  attendance_percentage        1000 non-null   int64  
 11  sleep_hours                  1000 non-null   float64
 12  stress_level                 1000 non-null   int64  
 13  internship         

In [189]:
# Consistency Score
df["consistency_score"] = (
    0.4 * (df["study_hours_per_day"] / 8) +
    0.3 * (df["dsa_problems_solved"] / 300).clip(upper=1) +
    0.3 * (df["attendance_percentage"] / 100)
)

# Effort Score
df["effort_score"] = df["study_hours_per_day"] * df["consistency_score"]

# Stress-Sleep Ratio
df["stress_sleep_ratio"] = (
    df["stress_level"] / df["sleep_hours"].replace(0, np.nan)
).fillna(df["stress_level"].median())

# Burnout Score
df["burnout_score"] = (
    0.5 * df["stress_level"] +
    0.3 * (8 - df["sleep_hours"]) +
    0.2 * (1 - df["consistency_score"]) * 10
)

# Practice Growth Rate
df["practice_growth_rate"] = (
    df["dsa_problems_solved"] /
    (df["study_hours_per_day"].replace(0, np.nan) * 30)
).fillna(0)

# Preparation Index
df["prep_index"] = (
    0.25 * (df["cgpa"] / 10) +
    0.20 * (df["mock_test_score"] / 100) +
    0.15 * (df["projects_count"].clip(upper=5) / 5) +
    0.15 * (df["resume_score"] / 100) +
    0.15 * df["consistency_score"] +
    0.10 * (df["core_subjects_confidence"] / 5)
)

# Risk Flags (VERY IMPORTANT)
df["low_cgpa_flag"] = (df["cgpa"] < 6.5).astype(int)
df["high_stress_flag"] = (df["stress_level"] > 7).astype(int)
df["low_dsa_flag"] = (df["dsa_problems_solved"] < 150).astype(int)
df["low_sleep_flag"] = (df["sleep_hours"] < 5).astype(int)

df.head()

,branch,gender,college_type,study_mode,cgpa,dsa_problems_solved,mock_test_score,projects_count,resume_score,study_hours_per_day,...,consistency_score,effort_score,stress_sleep_ratio,burnout_score,practice_growth_rate,prep_index,low_cgpa_flag,high_stress_flag,low_dsa_flag,low_sleep_flag
0,EEE,Female,Tier1,Coaching,5.63,380,70,3,59,1.9,...,0.563,1.0697,0.394737,2.494,6.666667,0.6037,1,0,0,0
1,MECH,Female,Tier1,Mixed,5.72,290,61,1,96,8.9,...,0.930,8.2770,0.500000,2.340,1.086142,0.5985,1,0,0,1
2,ECE,Female,Tier2,Self,6.11,579,44,3,89,5.7,...,0.789,4.4973,0.142857,1.222,3.385965,0.6626,1,0,0,0
3,MECH,Female,Tier3,Coaching,6.42,532,91,4,80,8.4,...,0.984,8.2656,1.097561,4.472,2.111111,0.7901,1,1,0,0
4,MECH,Female,Tier1,Mixed,8.36,236,56,5,58,8.7,...,0.836,7.2732,0.689655,2.988,0.904215,0.7234,0,0,0,0


In [190]:
df.describe()

,cgpa,dsa_problems_solved,mock_test_score,projects_count,resume_score,study_hours_per_day,attendance_percentage,sleep_hours,stress_level,internship,...,consistency_score,effort_score,stress_sleep_ratio,burnout_score,practice_growth_rate,prep_index,low_cgpa_flag,high_stress_flag,low_dsa_flag,low_sleep_flag
count,1000.000000,1000.000000,1000.00000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,...,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,7.690610,328.294000,64.68600,3.450000,67.235000,5.592400,74.528000,6.490400,5.115000,0.411000,...,0.746902,4.521124,0.825357,3.516576,2.759725,0.686845,0.217000,0.240000,0.173000,0.190000
std,1.222439,160.532658,20.31443,2.298431,18.781974,2.610996,14.108305,1.422431,2.593936,0.492261,...,0.158835,2.736740,0.465904,1.388338,2.662409,0.088495,0.412409,0.427297,0.378437,0.392497
min,5.510000,50.000000,30.00000,0.000000,35.000000,1.000000,50.000000,4.000000,1.000000,0.000000,...,0.285000,0.313500,0.112360,0.272000,0.173469,0.416250,0.000000,0.000000,0.000000,0.000000
25%,6.670000,186.750000,47.00000,1.000000,51.000000,3.400000,63.000000,5.300000,3.000000,0.000000,...,0.630750,2.115150,0.449438,2.405000,1.121863,0.625988,0.000000,0.000000,0.000000,0.000000
50%,7.720000,327.500000,65.00000,3.000000,67.000000,5.700000,74.000000,6.500000,5.000000,0.000000,...,0.747000,4.237500,0.800000,3.552000,1.942133,0.689950,0.000000,0.000000,0.000000,0.000000
75%,8.745000,475.000000,82.00000,5.000000,83.250000,7.900000,87.000000,7.700000,7.000000,1.000000,...,0.875000,6.800675,1.111111,4.632000,3.278846,0.749975,0.000000,0.000000,0.000000,0.000000
max,9.790000,599.000000,99.00000,7.000000,99.000000,10.000000,99.000000,9.000000,9.000000,1.000000,...,1.086000,10.751400,2.195122,6.720000,19.200000,0.938750,1.000000,1.000000,1.000000,1.000000


In [191]:
numeric_features = df.select_dtypes(include=[np.number])
categorical_features = df.select_dtypes(include=[object])
print("Numeric Features:\n", numeric_features.columns)
print("\nCategorical Features:\n", categorical_features.columns)

Numeric Features:
 Index(['cgpa', 'dsa_problems_solved', 'mock_test_score', 'projects_count',
       'resume_score', 'study_hours_per_day', 'attendance_percentage',
       'sleep_hours', 'stress_level', 'internship', 'core_subjects_confidence',
       'core_subjects_covered_count', 'consistency_score', 'effort_score',
       'stress_sleep_ratio', 'burnout_score', 'practice_growth_rate',
       'prep_index', 'low_cgpa_flag', 'high_stress_flag', 'low_dsa_flag',
       'low_sleep_flag'],
      dtype='object')

Categorical Features:
 Index(['branch', 'gender', 'college_type', 'study_mode', 'placement_status'], dtype='object')


In [192]:
df['placement_status'].value_counts() #balanced dataset or not checking 

placement_status
Placement Ready      580
Needs Improvement    405
At Risk               15
Name: count, dtype: int64

# according to above obesrvation the dataset is imbalanced so among undersampling, oversampling, smote, ensambling i chose smote to make it balanced
#👉 SMOTE improves learning, not evaluation
# imbalanced-learn (library)
# 👉 SMOTE is NOT a standalone library
# It is a class inside a library called imbalanced-learn
#to apply we need all our dataset values to be numerical

In [193]:
X = df.drop("placement_status", axis=1)
y = df["placement_status"]

In [194]:
from sklearn.preprocessing import LabelEncoder

In [195]:
le = LabelEncoder()
y = le.fit_transform(y)

In [196]:
X = pd.get_dummies(X, drop_first=True)

<!-- applied one-hot encoding to categorical input features and label encoding to the target variable since it is a multi-class classification problem. This ensured all inputs to the model were numerical -->

In [197]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [198]:
pip install imbalanced-learn

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'd:\Student_Placement_Readiness_Predictor\venv\Scripts\python.exe -m pip install --upgrade pip' command.


In [199]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_train, y_train = smote.fit_resample(X_train, y_train)


In [200]:

pd.Series(y_train).value_counts() #Since y_train is usually a NumPy array now

2    464
1    464
0    464
Name: count, dtype: int64

In [201]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [202]:
from sklearn.ensemble import RandomForestClassifier
#telling model to care about the at risk class also 
model = RandomForestClassifier(class_weight='balanced', random_state=42)

In [203]:
model.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [204]:
y_pred = model.predict(X_test)

In [205]:
from sklearn.metrics import accuracy_score

print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.91


In [206]:
from sklearn.metrics import confusion_matrix

print(confusion_matrix(y_test, y_pred))

[[  1   2   0]
 [  0  72   9]
 [  0   7 109]]


from above confusion matrix  model completely ignores class 0 because if we see clearly for class 0 ehich is at risk class it predicted actual values 0 (👉 Rows = Actual values
👉 Columns = Predicted values)

In [207]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      0.33      0.50         3
           1       0.89      0.89      0.89        81
           2       0.92      0.94      0.93       116

    accuracy                           0.91       200
   macro avg       0.94      0.72      0.77       200
weighted avg       0.91      0.91      0.91       200



In [208]:
from sklearn.linear_model import LogisticRegression

log_model = LogisticRegression()
log_model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [209]:
y_Lpred = log_model.predict(X_test)

In [210]:
from sklearn.metrics import accuracy_score

print("Accuracy:", accuracy_score(y_test, y_Lpred))

Accuracy: 0.98


In [211]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_Lpred)
print(cm)

[[  3   0   0]
 [  0  77   4]
 [  0   0 116]]


In [212]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_Lpred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00         3
           1       1.00      0.95      0.97        81
           2       0.97      1.00      0.98       116

    accuracy                           0.98       200
   macro avg       0.99      0.98      0.99       200
weighted avg       0.98      0.98      0.98       200



In [213]:
from sklearn.tree import DecisionTreeClassifier

dt_model = DecisionTreeClassifier()
dt_model.fit(X_train, y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,None
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [214]:
y_pred = dt_model.predict(X_test)

In [215]:
from sklearn.metrics import accuracy_score

print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.815


In [216]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)
print(cm)

[[  3   0   0]
 [  1  60  20]
 [  0  16 100]]


In [217]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.75      1.00      0.86         3
           1       0.79      0.74      0.76        81
           2       0.83      0.86      0.85       116

    accuracy                           0.81       200
   macro avg       0.79      0.87      0.82       200
weighted avg       0.81      0.81      0.81       200

